# Boosting

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
from matplotlib import pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification, make_moons, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

## AdaBoost custom implementation

### *Pseudo code*

input: $$ X^N, y, T $$
output: $$ b_1(x), ..., b_T(x); \alpha_1, ..., \alpha_T $$

logic:
1.  let $$ w_i = 1/N $$
2.  for each t in 1 .. T:
3.  $$ b_t = arg min N(b) $$
4.  $$ \alpha_t = \frac{1}{2} \ln {\frac{1 - N(b)}{N(b)}} $$
5.  $$ normalize(w) $$

In [ ]:
# task 1: implement adaboost_fit function for classification
#  base_model = DecisionTreeClassifier(max_depth=1)

In [ ]:
from typing import List, Tuple

def adaboost_fit(X, y, T) -> Tuple[List[DecisionTreeClassifier], List[float]]:
    ...

In [ ]:
# task 2: implement adaBoost_predict

In [ ]:
def adaboost_predict(X, models: List[DecisionTreeClassifier], alphas: List[float]):
    ...

In [ ]:
def plot_decision_boundaries(X, y, predict_func, title="Decision boundary", resolution=300):
    """
    Визуализация областей классификации.
    """
    x_min, x_max = X[:, 0].min() - 0.7, X[:, 0].max() + 0.7
    y_min, y_max = X[:, 1].min() - 0.7, X[:, 1].max() + 0.7

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, resolution),
        np.linspace(y_min, y_max, resolution)
    )

    X_grid = np.c_[xx.ravel(), yy.ravel()]
    Z = predict_func(X_grid)
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, alpha=0.3)
    plt.contour(xx, yy, Z, levels=[0], linewidths=1.5)

    plt.scatter(
        X[:, 0], X[:, 1],
        c=y,
        s=35,
        edgecolor="black"
    )

    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()


In [ ]:
X, y = make_classification(n_samples=500, n_features=10, random_state=42)
y = 2 * y - 1  # перевод в -1, 1

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models, alphas = adaboost_fit(X_train, y_train, T=50)

y_pred = adaboost_predict(X_test, models, alphas)
print("Точность:", accuracy_score(y_test, y_pred))

In [ ]:
X, y = make_moons(n_samples=400, noise=0.25, random_state=42)

# Переводим {0, 1} в {-1, +1}
y_pm = np.where(y == 0, -1, 1)

# Делим на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_pm, test_size=0.3, random_state=42
)

# Обучаем собственный AdaBoost
models, alphas = adaboost_fit(
    X_train,
    y_train,
    T=30,
)

# Предсказания
y_train_pred = adaboost_predict(X_train, models, alphas)
y_test_pred = adaboost_predict(X_test, models, alphas)

print("Train accuracy:", accuracy_score(y_train, y_train_pred))
print("Test accuracy: ", accuracy_score(y_test, y_test_pred))

# Визуализация границы решений на train
plot_decision_boundaries(
    X_train,
    y_train,
    predict_func=lambda X_new: adaboost_predict(X_new, models, alphas),
    title="AdaBoost (custom) on make_moons - train"
)

# Визуализация границы решений на test
plot_decision_boundaries(
    X_test,
    y_test,
    predict_func=lambda X_new: adaboost_predict(X_new, models, alphas),
    title="AdaBoost (custom) on make_moons - train"
)

In [ ]:
# task 3:
# visualize test error vs train error as a function of T

In [ ]:
def plot_error_vs_T(X_train, y_train, X_test, y_test, max_T: int = 100):
    ...
    plt.show()

In [ ]:
plot_error_vs_T(X_train, y_train, X_test, y_test, max_T=100)

## Boosting with sklearn, XGBoost and CatBoost

In [ ]:
!pip install xgboost catboost -q

In [ ]:
from time import perf_counter

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = data.target.copy()

df = X.copy()
df["target"] = y

df.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
def run_model(model, X_train, y_train, X_test, y_test, model_name):
    start = perf_counter()
    model.fit(X_train, y_train)
    fit_time = perf_counter() - start

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = y_pred

    metrics = {
        "model": model_name,
        "fit_time_sec": fit_time,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_score),
    }

    return metrics, y_pred, y_score

### AdaBoost

In [ ]:
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42), # базовый алгоритм классификации
    n_estimators=200,                                               # количество деревьев T
    random_state=42
)

adaboost_metrics, adaboost_pred, adaboost_score = run_model(
    ada_model, X_train, y_train, X_test, y_test, "AdaBoost"
)

adaboost_metrics

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,       # количество деревьев T
    learning_rate=0.05,     # шаг градиентного бустинга
    max_depth=3,            # максимальная глубина дерева
    subsample=0.9,          # доля объектов для каджого дерева
    colsample_bytree=0.9,   # доля признаков для каждого дерева
    reg_lambda=3.0,         # L-2 регуляризация
    eval_metric="logloss",  # функция потерь/оценки качества
    random_state=42
)

xgboost_metrics, xgboost_pred, xgboost_score = run_model(
    xgb_model, X_train, y_train, X_test, y_test, "XGBoost"
)

xgboost_metrics

In [ ]:
catboost_model = CatBoostClassifier(
    iterations=200,             # количество итераций
    learning_rate=0.05,         # шаг градиентного бустинга
    depth=4,                    # глубина деревьев
    l2_leaf_reg=3.0,            # L2-регуляризация для листьев
    loss_function="Logloss",    # функция потерь
    eval_metric="F1",           # функция оценки качества
    random_seed=42
)

catboost_metrics, catboost_pred, catboost_score = run_model(
    catboost_model, X_train, y_train, X_test, y_test, "CatBoost"
)

catboost_metrics

In [ ]:
results = pd.DataFrame([adaboost_metrics, xgboost_metrics, catboost_metrics]).sort_values(
    by="roc_auc", ascending=False
)

results.set_index("model")[["accuracy", "precision", "recall", "f1", "roc_auc"]].plot(
    kind="bar",
    figsize=(10, 5)
)
plt.title("Boosting models comparison on breast cancer dataset")
plt.ylabel("score")
plt.ylim(0.9, 1.01)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))

RocCurveDisplay.from_predictions(y_test, adaboost_score, name="AdaBoost", ax=plt.gca())
RocCurveDisplay.from_predictions(y_test, xgboost_score, name="XGBoost", ax=plt.gca())
RocCurveDisplay.from_predictions(y_test, catboost_score, name="CatBoost", ax=plt.gca())

plt.title("ROC curves")
plt.grid(alpha=0.3)
plt.show()

## Customer retention with IBM Telco customer churn data
https://www.kaggle.com/datasets/blastchar/telco-customer-churn

In [ ]:
df = pl.read_csv('./telco-customer-churn.csv')

print(df.shape)
df.head()

In [ ]:
df.describe()

In [ ]:
import polars as pl

new_df = (
    df
    # Убираем идентификатор
    .drop("customerID")
    .with_columns(
        # Преобразуем TotalCharges в число
        pl.col("TotalCharges").cast(pl.Float64, strict=False),
        # Целевая переменная
        pl.col("Churn").replace({"Yes": 1, "No": 0}).cast(pl.Int8)
    )
)

print(new_df.shape)
new_df.head()

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
X = new_df.drop("Churn")
y = new_df["Churn"]

X_pd = X.to_pandas()
y_pd = y.to_pandas()

categorical_features = X_pd.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X_pd.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
logreg_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

svm_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", SVC(
            kernel="rbf",
            C=1.0,
            probability=True,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

catboost_model = CatBoostClassifier(
    iterations=400,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    verbose=0,
    random_seed=42
)

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name, fit_params=None):
    if fit_params is None:
        fit_params = {}

    start = perf_counter()
    model.fit(X_train, y_train, **fit_params)
    fit_time = perf_counter() - start

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        raise ValueError(f"{model_name} does not support predict_proba")

    metrics = {
        "model": model_name,
        "fit_time_sec": round(fit_time, 3),
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_score),
        "pr_auc": average_precision_score(y_test, y_score)
    }

    return metrics, y_pred, y_score

In [ ]:
results = []
predictions = {}
scores = {}
trained_models = {}

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pd, y_pd,
    test_size=0.25,
    random_state=42,
    stratify=y_pd
)

In [ ]:
# Logistic Regression

LOGISTIC_REGRESSION = "Logistic Regression"

logreg_metrics, logreg_pred, logreg_score = evaluate_model(
    logreg_model, X_train, y_train, X_test, y_test, LOGISTIC_REGRESSION
)

results.append(logreg_metrics)
predictions[LOGISTIC_REGRESSION] = logreg_pred
scores[LOGISTIC_REGRESSION] = logreg_score
trained_models[LOGISTIC_REGRESSION] = logreg_model

logreg_metrics

In [ ]:
SVM_RBF = "SVM (RBF)"

svm_metrics, svm_pred, svm_score = evaluate_model(
    svm_model, X_train, y_train, X_test, y_test, SVM_RBF
)

results.append(svm_metrics)
predictions[SVM_RBF] = svm_pred
scores[SVM_RBF] = svm_score
trained_models[SVM_RBF] = svm_model

svm_metrics

In [ ]:
RANDOM_FOREST = "Random Forest"

rf_metrics, rf_pred, rf_score = evaluate_model(
    rf_model, X_train, y_train, X_test, y_test, RANDOM_FOREST
)

results.append(rf_metrics)
predictions[RANDOM_FOREST] = rf_pred
scores[RANDOM_FOREST] = rf_score
trained_models[RANDOM_FOREST] = rf_model

rf_metrics

In [ ]:
CATBOOST = "CatBoost"

cat_features_idx = [X_train.columns.get_loc(col) for col in categorical_features]

cat_metrics, cat_pred, cat_score = evaluate_model(
    catboost_model,
    X_train,
    y_train,
    X_test,
    y_test,
    CATBOOST,
    fit_params={"cat_features": cat_features_idx}
)

results.append(cat_metrics)
predictions[CATBOOST] = cat_pred
scores[CATBOOST] = cat_score
trained_models[CATBOOST] = catboost_model

cat_metrics

In [ ]:
results_df = pd.DataFrame(results).sort_values(by="roc_auc", ascending=False)
results_df

In [ ]:
metric_cols = ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]

results_df.set_index("model")[metric_cols].plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Model comparison on churn prediction")
plt.ylabel("score")
plt.xticks(rotation=20)
plt.ylim(0.0, 1.05)
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

for model_name in scores:
    RocCurveDisplay.from_predictions(
        y_test,
        scores[model_name],
        name=model_name,
        ax=plt.gca()
    )

plt.title("ROC curves")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
cv_results = []

In [ ]:
logreg_cv_scores = cross_val_score(
    logreg_model,
    X_pd,
    y_pd,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

cv_results.append({
    "model": "Logistic Regression",
    "cv_metric": "roc_auc",
    "mean": logreg_cv_scores.mean(),
    "std": logreg_cv_scores.std(),
    "scores": logreg_cv_scores
})

logreg_cv_scores

In [ ]:
svm_cv_scores = cross_val_score(
    svm_model,
    X_pd,
    y_pd,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

cv_results.append({
    "model": "SVM (RBF)",
    "cv_metric": "roc_auc",
    "mean": svm_cv_scores.mean(),
    "std": svm_cv_scores.std(),
    "scores": svm_cv_scores
})

svm_cv_scores

In [ ]:
rf_cv_scores = cross_val_score(
    rf_model,
    X_pd,
    y_pd,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

cv_results.append({
    "model": "Random Forest",
    "cv_metric": "roc_auc",
    "mean": rf_cv_scores.mean(),
    "std": rf_cv_scores.std(),
    "scores": rf_cv_scores
})

rf_cv_scores

In [ ]:
cat_features_idx = [X_pd.columns.get_loc(col) for col in categorical_features]

cat_cv_scores = []

for train_idx, valid_idx in cv.split(X_pd, y_pd):
    X_train_cv = X_pd.iloc[train_idx]
    X_valid_cv = X_pd.iloc[valid_idx]
    y_train_cv = y_pd.iloc[train_idx]
    y_valid_cv = y_pd.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=400,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3.0,
        loss_function="Logloss",
        eval_metric="AUC",
        auto_class_weights="Balanced",
        verbose=0,
        random_seed=42
    )

    model.fit(
        X_train_cv,
        y_train_cv,
        cat_features=cat_features_idx
    )

    y_valid_score = model.predict_proba(X_valid_cv)[:, 1]
    fold_auc = roc_auc_score(y_valid_cv, y_valid_score)
    cat_cv_scores.append(fold_auc)

cat_cv_scores = np.array(cat_cv_scores)

cv_results.append({
    "model": "CatBoost",
    "cv_metric": "roc_auc",
    "mean": cat_cv_scores.mean(),
    "std": cat_cv_scores.std(),
    "scores": cat_cv_scores
})

cat_cv_scores

In [ ]:
cv_results_df = pd.DataFrame(cv_results)[["model", "cv_metric", "mean", "std"]]
cv_results_df = cv_results_df.sort_values("mean", ascending=False)
cv_results_df

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    cv_results_df["model"],
    cv_results_df["mean"],
    yerr=cv_results_df["std"],
    capsize=5
)

plt.title("5-fold cross-validation ROC-AUC")
plt.ylabel("ROC-AUC")
plt.ylim(0.7, 1.0)
plt.grid(axis="y", alpha=0.3)
plt.xticks(rotation=20)
plt.show()